# Proyecto Integrador — Semana 02  
# 01 Bronze — Daniel Guzmán

## Objetivo

Ingestar las 5 fuentes del dataset financiero como tablas Delta en la capa Bronze, manteniendo los datos lo más fieles posible a la fuente.

## Fuentes

- `transactions_data.csv`
- `users_data.csv`
- `cards_data.csv`
- `mcc_codes.json`
- `train_fraud_labels.parquet`

## Salidas Bronze

- `workspace.bronze.transactions_daniel`
- `workspace.bronze.users_daniel`
- `workspace.bronze.cards_daniel`
- `workspace.bronze.mcc_codes_daniel`
- `workspace.bronze.fraud_labels_daniel`

## Nota técnica

El archivo de fraude también existe como JSON, pero en Databricks Free/Serverless puede tardar demasiado.  
Para este proyecto se usa `train_fraud_labels.parquet`, documentando la decisión por performance.

La capa Bronze conserva los datos de la forma más fiel posible. Solo se agrega la columna técnica `_ingested_at` para registrar el momento de ingesta.

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print(f"Volume usado: {VOL}")
display(dbutils.fs.ls(VOL))

In [0]:
# BRONZE — Ingesta sin transformaciones de negocio
# Solo se agrega _ingested_at como columna técnica de auditoría.

# 1. transactions
df_bronze_tx = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/transactions_data.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze_tx.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.transactions_{MI_NOMBRE}")

print(f"{CATALOG}.bronze.transactions_{MI_NOMBRE}: {df_bronze_tx.count():,} filas | {len(df_bronze_tx.columns)} columnas")


# 2. users
df_bronze_users = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/users_data.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze_users.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.users_{MI_NOMBRE}")

print(f"{CATALOG}.bronze.users_{MI_NOMBRE}: {df_bronze_users.count():,} filas | {len(df_bronze_users.columns)} columnas")


# 3. cards
df_bronze_cards = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/cards_data.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze_cards.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.columnMapping.mode", "name") \
    .saveAsTable(f"{CATALOG}.bronze.cards_{MI_NOMBRE}")

print(f"{CATALOG}.bronze.cards_{MI_NOMBRE}: {df_bronze_cards.count():,} filas | {len(df_bronze_cards.columns)} columnas")


# 4. mcc_codes
df_bronze_mcc = (
    spark.read
    .option("multiLine", "true")
    .json(f"{VOL}/mcc_codes.json")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze_mcc.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}")

print(f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}: {df_bronze_mcc.count():,} filas | {len(df_bronze_mcc.columns)} columnas")


# 5. fraud_labels
df_bronze_fraud = (
    spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_bronze_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}")

print(f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}: {df_bronze_fraud.count():,} filas | {len(df_bronze_fraud.columns)} columnas")


print("\nBronze completo. Tablas disponibles:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.bronze"))

## Documentación Bronze

Se ingestaron las 5 fuentes del dataset financiero como tablas Delta en el schema `bronze`.

Tablas creadas:

- `workspace.bronze.transactions_daniel`
- `workspace.bronze.users_daniel`
- `workspace.bronze.cards_daniel`
- `workspace.bronze.mcc_codes_daniel`
- `workspace.bronze.fraud_labels_daniel`

La capa Bronze se mantuvo fiel a la fuente. No se hicieron transformaciones de negocio, renombrados ni limpieza de tipos.

La única columna agregada fue `_ingested_at`, usada como campo técnico de auditoría para registrar el momento de ingesta.

Para `fraud_labels` se usó `train_fraud_labels.parquet` en lugar de `train_fraud_labels.json`, porque en Databricks Free/Serverless el JSON tarda demasiado en procesarse. Esta decisión permite continuar el pipeline sin alterar el contenido analítico necesario.